# ATHENA–CAMM Hackathon | Ternary Spectroscopy
## Two challenges, one hidden measurement API

**Challenge 1 — Learn to measure a spectrum:** At one fixed composition, choose wavelengths and acquisition durations to reconstruct the complete **512-channel spectrum**.

**Challenge 2 — Explore the whole ternary wafer:** Choose both compositions and wavelengths/time to reconstruct the full **spectral cube** $S(x,y,E)$ across the ternary triangle.

**Scientific objective:** Minimize reconstruction error compared with organizer-only hidden noiseless ground truth. **Phase classification is not the scoring target.** The true phase diagram, true spectra, and noise laws are not visible to participants.

The following baselines are intentionally simple. Replace their acquisition and interpolation strategies with your own. Running the acquisition cells consumes episode budget.

## 0. Connect to the common HTTPS API

Use the actual URL from the organizers. The API must have the **two-challenge backend** deployed; the original single-task API will not work for Challenge 2.

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

BASE_URL = "https://athena-spectral-api.onrender.com"  # Replace if organizer provides a different URL
BASE_URL = BASE_URL.rstrip("/")

def api(method, path, payload=None, token=None):
    headers = {"X-Session-ID": token} if token else {}
    response = requests.request(method, BASE_URL + path, json=payload, headers=headers, timeout=120)
    if not response.ok:
        raise RuntimeError(f"{method} {path}: HTTP {response.status_code}: {response.text[:500]}")
    return response.json()

info = api("GET", "/info")
print("Available tasks:", info["tasks"])
print("Single-spectrum budget:", info["time_budget"])
print("Full-wafer budget:", info["cube_time_budget"])
print("Evaluation cube shape:", info["cube_shape"])
energy = np.linspace(*info["wavelength_range"], info["channels"])


---
# Challenge 1 — Optimal measurement of a single spectrum

**Goal:** At fixed $(x,y)$, decide the wavelength $E_i$ and acquisition duration $t_i$ of each scalar measurement. Return one prediction for each of 512 energy channels. Initial budget: **30 acquisition-time units**, intended for roughly **20–40 measurements** (depending on your chosen durations).

Only `/pixel` is used. Full-spectrum acquisitions are deliberately unavailable in the benchmark to make wavelength selection meaningful.

In [ ]:
episode1 = api("POST", "/session", {"challenge": 1})
TOKEN1 = episode1["session_id"]
x1, y1 = info["composition"]["x"], info["composition"]["y"]
print(f"Episode 1 at (x,y,z)=({x1:.2f}, {y1:.2f}, {1-x1-y1:.2f})")
print("Budget:", episode1["time_budget"])

def measure1(wavelength, time):
    return api("POST", "/pixel", {"wavelength":float(wavelength), "time":float(time)}, TOKEN1)

def budget1():
    return api("GET", "/budget", token=TOKEN1)


## Challenge 1 baseline — 30 uniformly spaced spectral measurements

This cell **spends the full Challenge 1 budget**. Do not rerun it unless you intend to start a new episode. A better method could sample adaptively near uncertain features and redistribute acquisition time.

In [ ]:
n1 = 30
sample_E1 = np.linspace(energy.min(), energy.max(), n1)
t1 = episode1["time_budget"] / n1
observations1 = [measure1(w, t1) for w in sample_E1]
measured_E1 = np.array([r["wavelength"] for r in observations1])
measured_I1 = np.array([r["intensity"] for r in observations1])
predicted_spectrum1 = np.interp(energy, measured_E1, measured_I1)
print("Measurement count:",len(observations1),"|",budget1())


In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(energy, predicted_spectrum1, lw=2, label="Predicted 512-channel spectrum")
ax.scatter(measured_E1, measured_I1, s=25, c="tab:red", zorder=3, label="Actual measurements")
ax.set(xlabel="Spectral coordinate (a.u.)",ylabel="Intensity (a.u.)",title="Challenge 1 | Observations and reconstruction")
ax.legend(); plt.tight_layout(); plt.show()


## Challenge 1 submission

Organizers evaluate RMSE against the hidden clean spectrum. Submitting locks this episode. **Uncomment to submit.**

In [ ]:
submission1 = {"energy": energy.tolist(), "predicted_intensity": predicted_spectrum1.tolist()}
# score1 = api("POST", "/submit", submission1, TOKEN1)
# print("Challenge 1 score:",score1)


---
# Challenge 2 — Optimal exploration of the full ternary spectral landscape

**Goal:** Reconstruct $S(x,y,E)$ for every composition on an organizer-defined triangular evaluation grid and every one of 512 spectral channels. The agent chooses **where** to measure, **which wavelengths** to measure there, and **how long** to acquire each observation.

The initial budget is **900 time units** (30× the single-spectrum budget), pending organizer calibration. The returned grid is an **evaluation grid**, not a measurement grid requirement. Measurements may be requested at any valid composition $x\ge0$, $y\ge0$, $x+y\le1$.

The baseline below samples 24 compositions and 12 wavelengths per composition (288 scalar measurements). It reconstructs spectra by 1D interpolation, followed by 2D interpolation over composition. This is only a starting point; it can miss narrow peaks and sharp phase boundaries.

In [ ]:
episode2 = api("POST", "/session", {"challenge": 2})
TOKEN2 = episode2["session_id"]
coords_eval = np.asarray(info["cube_grid"], dtype=float)
assert coords_eval.shape[1] == 2
assert len(energy) == info["cube_shape"][1]

def measure2(x, y, wavelength, time):
    return api("POST", "/pixel", {"x":float(x),"y":float(y),
                  "wavelength":float(wavelength),"time":float(time)}, TOKEN2)

def budget2():
    return api("GET", "/budget", token=TOKEN2)

print("Evaluation locations:",len(coords_eval),"| Budget:",episode2["time_budget"])


## Challenge 2 baseline — Sparse ternary locations and sparse spectra

A triangular **six-level** measurement grid has 21 unique compositions. We add three interior points for **24 compositions total**, with 12 wavelengths per composition. All observations share the budget equally.

**Warning:** This cell makes **288 API calls** and consumes the episode's entire measurement budget. It may take some time on a remote server. For a quick connectivity test, measure one point in a fresh test episode instead of running the baseline.

In [ ]:
level = 6
coarse_coords = np.array([(i/(level-1),j/(level-1))
                          for i in range(level) for j in range(level-i)], dtype=float)
extra_coords = np.array([[.24,.37],[.42,.24],[.28,.20]])
train_coords = np.vstack([coarse_coords, extra_coords])
nE = 12
train_energy = np.linspace(energy.min(), energy.max(), nE)
per_read_time = episode2["time_budget"] / (len(train_coords)*nE)

sample_matrix = np.empty((len(train_coords), nE))
for i,(x,y) in enumerate(train_coords):
    for j,w in enumerate(train_energy):
        sample_matrix[i,j] = measure2(x,y,w,per_read_time)["intensity"]
print("Measurements:",sample_matrix.size,"|",budget2())


## Reconstruct the complete cube

**Stage A:** Interpolate each measured 12-channel spectrum to all 512 channels.

**Stage B:** At each channel, interpolate those estimates spatially onto the prescribed triangular evaluation grid. Use nearest-neighbor estimates where linear interpolation is undefined. Output shape is `(N_compositions, 512)`; rows must follow the order of `info["cube_grid"]`.

In [ ]:
spectra_at_measured_coords = np.array([
    np.interp(energy, train_energy, spectrum) for spectrum in sample_matrix
])
predicted_cube = np.empty((len(coords_eval), len(energy)), dtype=float)
for k in range(len(energy)):
    linear = griddata(train_coords, spectra_at_measured_coords[:,k], coords_eval, method="linear")
    missing = np.isnan(linear)
    if np.any(missing):
        linear[missing] = griddata(train_coords, spectra_at_measured_coords[:,k],
                                  coords_eval[missing], method="nearest")
    predicted_cube[:,k] = linear
assert predicted_cube.shape == tuple(info["cube_shape"])
print("Predicted spectral cube:", predicted_cube.shape)


## Visualize what you actually predict

The ternary plots below show **predicted spectral intensity at three energies**, not the hidden phase labels. Black circles show sampled compositions. A second plot compares reconstructed spectra at three selected evaluation compositions. These are the required scientific outputs for Challenge 2. A phase map is optional and would require an additional phase-inference model.

In [ ]:
from matplotlib.tri import Triangulation

def to_triangle(xy):
    x,y = xy[:,0],xy[:,1]
    c = 1-x-y
    return y + 0.5*c, np.sqrt(3)/2*c

X,Y = to_triangle(coords_eval)
Xm,Ym = to_triangle(train_coords)
tri = Triangulation(X,Y)
fig, axes = plt.subplots(1,3,figsize=(15,4.6),constrained_layout=True)
for ax,E0 in zip(axes,[2.0,4.8,7.2]):
    idx = np.argmin(abs(energy-E0))
    im = ax.tricontourf(tri,predicted_cube[:,idx],levels=16,cmap="viridis")
    ax.scatter(Xm,Ym,s=13,edgecolors="white",facecolors="black",linewidths=.3,zorder=4)
    ax.plot([0,1,.5,0],[0,0,np.sqrt(3)/2,0],color="black",lw=1)
    ax.set_aspect("equal");ax.axis("off")
    ax.set_title(f"Predicted intensity | E={energy[idx]:.2f}")
    fig.colorbar(im,ax=ax,shrink=.72,label="Intensity (a.u.)")
plt.show()

fig,ax=plt.subplots(figsize=(9,4))
for target in [(0,0),(.3,.4),(0,1)]:
    i=np.argmin(np.sum((coords_eval-np.array(target))**2,axis=1))
    ax.plot(energy,predicted_cube[i],label=f"x={coords_eval[i,0]:.2f}, y={coords_eval[i,1]:.2f}")
ax.set(xlabel="Spectral coordinate (a.u.)",ylabel="Predicted intensity (a.u.)",
       title="Challenge 2 | Predicted spectra at selected compositions")
ax.legend();plt.tight_layout();plt.show()


## Challenge 2 submission

The organizer checks the predicted entire spectral cube against the clean hidden cube on the **fixed evaluation grid**. It returns RMSE and resource usage in this prototype. Submitting locks the episode. **Uncomment to submit.**

In [ ]:
submission2 = {"predictions": predicted_cube.tolist()}
# score2 = api("POST", "/submit-cube", submission2, TOKEN2)
# print("Challenge 2 score:",score2)


---
## Competition notes and integrity

- Both tasks use the **same `/pixel` measurement interface**; only the allowed composition controls and budget differ.
- Spectral wavelength here is a spectral coordinate in **arbitrary units**, not calibrated nanometers.
- A measurement consumes the acquisition time specified in its request; participant models may choose different wavelengths, compositions, and durations.
- **Challenge 1:** Score RMSE over 512 channels at the fixed composition.
- **Challenge 2:** Score RMSE over *all compositions and all channels* in `info["cube_grid"]`.
- Keep the original instructor notebook and the server-side simulator **private**.
- The provided baselines are not optimal. Implement adaptive design, spectral and spatial Gaussian processes, Bayesian approaches, or other methods to improve efficiency.
- **Organizer deployment caveat:** This is a prototype. Server state is held in memory; sessions can be restarted without limits, and final RMSE is returned to participants. Add persistent storage, team authentication, episode quotas, rate limiting, and a private final leaderboard before a scored event. Public read access to the API is intentional; public read access to the code is not.
- For a fair competition, standardize evaluation grid, budget, benchmark compositions, and final scoring conditions, and test performance across multiple hidden tasks if desired.